# Trip Concierge — Fine-Tuning the Booking Advisor (SFT)

Build a small **supervised fine-tuning (SFT)** dataset of near-booking turns labeled `book` vs
`dont_book`, launch an SFT job on `gpt-4o-mini-2024-07-18`, and compare the fine-tuned model against
the base model on a held-out split.

**The goal is to demonstrate the SFT workflow**, not to chase accuracy — Step 6 (few-shot prompt
tuning) already took the end-to-end eval to 97.1%. This notebook mirrors `test_evals.ipynb`: small,
numbered steps, one per cell.

> 🛠️ **This is a skeleton for you to fill in.** Code cells marked `# TODO` are yours to complete; the
> markdown above each cell says exactly what that step needs to do. The setup cells (1–4) are already
> written so you can focus on the fine-tuning steps.

> ⚠️ **Costs money and time.** A fine-tuning job uploads data, trains for several minutes, and bills
> for training tokens. Run the upload / launch / poll cells deliberately, not on autopilot.

## 1. Imports & project path

Same anchoring as the eval notebook: change into the **repo root** so the app's relative data paths
resolve, and put the root on `sys.path` so `from app...` imports work. The kernel starts in `tests/`,
so the repo root is one directory up (computed re-run-safely).

In [1]:
import json
import os
import sys
from pathlib import Path
from collections import Counter

# This notebook lives in tests/. Compute the repo root re-run-safely (after the chdir below,
# cwd is already the repo root, so guard on the folder name), then chdir into it.
_here = Path.cwd()
REPO_ROOT = _here.parent if _here.name == "tests" else _here
os.chdir(REPO_ROOT)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("repo root:", REPO_ROOT)

repo root: c:\Users\tomsa\source\repos\trip-concierge


## 2. Load environment variables

Load your `.env` so `OPENAI_API_KEY` is in the environment. The fine-tuning + files API read it the
same way the rest of the app does.

In [2]:
from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))

OPENAI_API_KEY set: True


## 3. Create the OpenAI client

Fine-tuning and file upload are SDK/REST features, so we use the official `openai` client directly
(LangChain only wraps chat). `OpenAI()` reads `OPENAI_API_KEY` from the environment automatically.

In [3]:
from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from the environment

## 4. The label space & system prompt

The Booking Advisor is a **binary** classifier: `book` (genuine commitment to one specific package) vs
`dont_book` (musing / question / hesitation).

We **import the live `SYSTEM` prompt** from `booking_advisor.py` rather than retyping it, so training uses
the *exact* instruction the advisor sends at inference — no train/serve skew, and no copy that can drift.
Note it's **long**: it carries the few-shot block added in Step 6, and `booking_advisor.py` sends that
same full prompt to *whichever* model is configured (the base model or the fine-tune).

> **Alternative (not taken here):** fine-tuning is often used to *replace* few-shot examples — you'd train
> on a short, instruction-only prompt so inference prompts get cheaper, since the model has internalized
> the task. That only pays off if the live advisor also sends the short prompt to the fine-tuned model, so
> it would mean editing `booking_advisor.py` too. We keep the prompts identical to stay consistent with the
> app as it is today.

In [4]:
LABELS = ("book", "dont_book")

# Train on the EXACT prompt the live Booking Advisor sends, so the model sees the same instruction
# at training and inference time (no train/serve skew). booking_advisor.py sends this full SYSTEM to
# whichever model is configured — base OR the fine-tune — so we IMPORT it rather than copy it, which
# means it can't drift. Heads-up: it's long — it includes the few-shot block added in step 6.
from app.modules.agents.booking_advisor.booking_advisor import SYSTEM

print(SYSTEM)

You are the Booking Advisor for a trip-planning concierge.

Decide whether the traveller has genuinely committed to booking ONE specific package,
or is only musing / asking / hesitating.

Reply with EXACTLY one word, lowercase, nothing else: book or dont_book.
- book: a clear commitment to a specific option ("book option 2", "let's do the Lisbon one").
- dont_book: musing, comparing, questions, or hesitation ("option 2 looks tempting…").

A commitment counts even when the wording is casual or implicit — what matters is that the
traveller is telling you to go ahead with one specific option, not still weighing them.

Examples — the traveller's latest message, then your answer:
- "Book option 2." -> book
- "Let's do the Lisbon one." -> book
- "Yes, go ahead and book it." -> book
- "I'll take the second package." -> book
- "Sounds great — lock in the Bali trip." -> book
- "Perfect, let's go with option 1." -> book
- "Option 2 looks tempting…" -> dont_book
- "How many nights is option 1?" -

## 5. Build the labeled dataset  ⟵ *fill this in*

Hand-craft near-booking turns, each a `(traveller's latest message, gold label)` pair. Aim for roughly
**~35 training** and **~9 held-out test** examples, balanced across the two classes.

Two things to cover deliberately:
- **`book`** — many *ways of committing* ("book option 2", "let's go with the Lisbon one", "I'll take
  the second package", "lock it in"). Variety here is what lifts recall.
- **`dont_book`** — musing, questions, comparisons, hesitation ("option 2 looks tempting…", "how many
  nights?", "can you compare 1 and 2?", "let me think about it").

Keep the test lines genuinely **held out** — don't reuse any training line.

In [5]:
# Hand-crafted, balanced dataset of near-booking turns: the traveller's latest message paired
# with the binary label the Booking Advisor must produce.
#
# Built to be DISJOINT from tests/trip_conversations.json (the eval set) AND from the few-shot
# block in booking_advisor.py's SYSTEM, so neither the end-to-end eval nor the in-prompt demos
# leak into training/test. `book` = genuine commitment to ONE specific package; `dont_book` =
# musing / question / comparison / hesitation about a specific option.

TRAIN = [
    # --- book: genuine commitment, varied phrasings ---
    ("Let's go ahead with option 3.", "book"),
    ("Book me the Barcelona one.", "book"),
    ("I'm sold — reserve option 1.", "book"),
    ("Yes please, confirm the Kyoto trip.", "book"),
    ("Great, let's lock in option 2.", "book"),
    ("Go ahead and book the third package.", "book"),
    ("That first option is perfect — book it for me.", "book"),
    ("Let's do it — reserve the Lisbon package.", "book"),
    ("Yes, finalize the Bali booking.", "book"),
    ("I'll take option 3.", "book"),
    ("Book the cheapest one for me.", "book"),
    ("Let's make it official — option 2, please.", "book"),
    ("Perfect, I'll go with the second package.", "book"),
    ("Reserve option 1 for me, thanks.", "book"),
    ("Yeah, let's book the Kyoto stay.", "book"),
    ("I want the Barcelona option — go ahead and book.", "book"),
    ("Done deliberating — book option 3.", "book"),
    ("Let's secure the Rome package.", "book"),
    # --- dont_book: musing / questions / comparison / hesitation ---
    ("Hmm, I'm torn between options 1 and 3.", "dont_book"),
    ("Is breakfast included in the first package?", "dont_book"),
    ("How far is the hotel from the beach?", "dont_book"),
    ("Are there any cheaper options than these?", "dont_book"),
    ("Option 3 looks nice, but I'm not ready yet.", "dont_book"),
    ("Can you tell me more about the second hotel?", "dont_book"),
    ("Do any of these depart in September instead?", "dont_book"),
    ("What's the difference between option 1 and option 3?", "dont_book"),
    ("Tempting, but I need to think about the budget.", "dont_book"),
    ("Is the flight direct on the Rome option?", "dont_book"),
    ("Could you hold option 2 while I decide?", "dont_book"),
    ("Not sure yet — what would you pick?", "dont_book"),
    ("How many nights is the Barcelona one again?", "dont_book"),
    ("Remind me what's included in option 3.", "dont_book"),
    ("I might go for the Kyoto trip, but not today.", "dont_book"),
    ("Are these prices per person or total?", "dont_book"),
    ("What's the weather like around those Lisbon dates?", "dont_book"),
]

TEST = [  # held out — no line here appears in TRAIN, the eval set, or the SYSTEM few-shot block
    ("I'd like to book the Rome stay, please.", "book"),
    ("Sign me up for option 2.", "book"),
    ("Option 1 it is. Book it.", "book"),
    ("Confirm the Reykjavik trip, the 7-night one.", "book"),
    ("I'm ready to book — option 1, please.", "book"),
    ("What's the cancellation policy on option 2?", "dont_book"),
    ("I'm leaning toward Bali, but let me check with my partner first.", "dont_book"),
    ("Which of these has the best reviews?", "dont_book"),
    ("Let me sleep on it before deciding.", "dont_book"),
]

# Sanity checks: no train/test overlap, plus sizes and class balance.
assert not (set(t for t, _ in TRAIN) & set(t for t, _ in TEST)), "TRAIN/TEST overlap!"
print(f"train={len(TRAIN)}  test={len(TEST)}")
print("train balance:", dict(Counter(label for _, label in TRAIN)))
print("test  balance:", dict(Counter(label for _, label in TEST)))

train=35  test=9
train balance: {'book': 18, 'dont_book': 17}
test  balance: {'book': 5, 'dont_book': 4}


## 6. Convert to OpenAI chat-format JSONL  ⟵ *fill this in*

The fine-tuning API wants one JSON object per line, each a chat conversation whose **final assistant
message is the target** the model learns to produce:

```json
{"messages": [
  {"role": "system", "content": "<SYSTEM>"},
  {"role": "user", "content": "Let's book option 2."},
  {"role": "assistant", "content": "book"}
]}
```

Implement `to_chat_example(text, label)` to return exactly that dict.

In [6]:
def to_chat_example(text, label):
    """Return one fine-tuning example in OpenAI chat format:
    {"messages": [ {system=SYSTEM}, {user=text}, {assistant=label} ]}.
    The assistant message is the TARGET the model learns to produce.
    """
    return {"messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": text},
        {"role": "assistant", "content": label}
    ]}

# Sanity check once implemented:
print(json.dumps(to_chat_example(*TRAIN[0]), indent=2))

{
  "messages": [
    {
      "role": "system",
      "content": "You are the Booking Advisor for a trip-planning concierge.\n\nDecide whether the traveller has genuinely committed to booking ONE specific package,\nor is only musing / asking / hesitating.\n\nReply with EXACTLY one word, lowercase, nothing else: book or dont_book.\n- book: a clear commitment to a specific option (\"book option 2\", \"let's do the Lisbon one\").\n- dont_book: musing, comparing, questions, or hesitation (\"option 2 looks tempting\u2026\").\n\nA commitment counts even when the wording is casual or implicit \u2014 what matters is that the\ntraveller is telling you to go ahead with one specific option, not still weighing them.\n\nExamples \u2014 the traveller's latest message, then your answer:\n- \"Book option 2.\" -> book\n- \"Let's do the Lisbon one.\" -> book\n- \"Yes, go ahead and book it.\" -> book\n- \"I'll take the second package.\" -> book\n- \"Sounds great \u2014 lock in the Bali trip.\" -> book\n-

## 7. Write the training JSONL  ⟵ *fill this in*

Write one JSON line per **training** example to `tests/finetune_data/booking_train.jsonl`. The TEST
split stays in memory — we score it ourselves in step 11 rather than handing it to the API.

In [7]:
FT_DIR = REPO_ROOT / "tests" / "finetune_data"
FT_DIR.mkdir(exist_ok=True)
TRAIN_PATH = FT_DIR / "booking_train.jsonl"

# TODO: write each to_chat_example(text, label) for TRAIN as its own json.dumps(...) line.
# Hint:
#   with open(TRAIN_PATH, "w", encoding="utf-8") as f:
#       for text, label in TRAIN:
#           f.write(json.dumps(to_chat_example(text, label)) + "\n")

with open(TRAIN_PATH, "w", encoding="utf-8") as f:
    for text, label in TRAIN:
        f.write(json.dumps(to_chat_example(text, label)) + "\n")
        
print("wrote:", TRAIN_PATH, "exists:", TRAIN_PATH.exists())

wrote: c:\Users\tomsa\source\repos\trip-concierge\tests\finetune_data\booking_train.jsonl exists: True


## 8. Upload the training file  ⟵ *fill this in*

Upload the JSONL with `purpose="fine-tune"` and keep the returned **file id** — the job in the next
step refers to it.

In [8]:
# TODO: upload TRAIN_PATH and capture the file id.
#   uploaded = client.files.create(file=open(TRAIN_PATH, "rb"), purpose="fine-tune")
#   training_file_id = uploaded.id

uploaded = client.files.create(file=open(TRAIN_PATH, "rb"), purpose="fine-tune")
training_file_id = uploaded.id  # <- replace
print("training_file_id:", training_file_id)

training_file_id: file-2tmEB82ZvXA5HdJeiyGZtN


## 9. Launch the fine-tuning job  ⟵ *fill this in*

Create the SFT job from your uploaded file on the base model. This **starts billing**. Keep the
returned **job id** for polling.

In [9]:
BASE_MODEL = "gpt-4o-mini-2024-07-18"

# TODO: create the fine-tuning job and capture its id.
#   job = client.fine_tuning.jobs.create(training_file=training_file_id, model=BASE_MODEL)
#   job_id = job.id

job = client.fine_tuning.jobs.create(training_file=training_file_id, model=BASE_MODEL)
job_id = job.id
print("job_id:", job_id)

job_id: ftjob-YGKiiKHjW3o8jdtSn29clA9o


## 10. Wait for the job to finish  ⟵ *fill this in*

Poll `client.fine_tuning.jobs.retrieve(job_id)` until the status leaves the queued/running states.
Training usually takes several minutes. When it `succeeded`, the `fine_tuned_model` field holds your
`ft:...` id. Don't tight-loop the API — sleep between checks (or just re-run this cell now and then).

In [10]:
import time

# TODO: poll until done, then read job.fine_tuned_model.
#   while True:
#       job = client.fine_tuning.jobs.retrieve(job_id)
#       print(job.status)
#       if job.status in ("succeeded", "failed", "cancelled"):
#           break
#       time.sleep(30)
#   ft_model = job.fine_tuned_model

while True:
    job = client.fine_tuning.jobs.retrieve(job_id)
    print(job.status)
    if job.status in ("succeeded", "failed", "cancelled"):
        break
    time.sleep(30)

ft_model = job.fine_tuned_model  # <- the ft:... id once the job succeeds
print("fine_tuned_model:", ft_model)

running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
succeeded
fine_tuned_model: ft:gpt-4o-mini-2024-07-18:personal::DuwBSoJZ


## 11. Evaluate base vs fine-tuned on the held-out split  ⟵ *fill this in*

The payoff: does SFT help on **unseen** near-booking turns? Implement `classify(model, text)` to send
`[SYSTEM, user=text]` to the chat API at `temperature=0` and return the one-word label, then score both
`BASE_MODEL` and `ft_model` on `TEST`.

In [11]:
def classify(model, text):
    """Send [SYSTEM, user=text] to `model` at temperature 0; return 'book' or 'dont_book'.

    Normalization mirrors booking_advisor.py's _classify (strip, drop a trailing period,
    lowercase, then anything that isn't exactly 'book' -> 'dont_book'), so we score each model
    the same way the live advisor would read its reply.
    """
    resp = client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": text},
        ],
    )
    label = resp.choices[0].message.content.strip().strip(".").lower()
    return "book" if label == "book" else "dont_book"


def accuracy_on(model):
    preds = [classify(model, text) for text, _ in TEST]
    correct = sum(p == gold for p, (_, gold) in zip(preds, TEST))
    return correct / len(TEST), preds


base_acc, base_preds = accuracy_on(BASE_MODEL)
ft_acc, ft_preds = accuracy_on(ft_model)

print(f"base     : {base_acc:.1%}  ({BASE_MODEL})")
print(f"fine-tune: {ft_acc:.1%}  ({ft_model})")

# Where did either model disagree with the gold label on the held-out split?
print("\nmisclassifications (gold | base | ft | text):")
any_miss = False
for (text, gold), bp, fp in zip(TEST, base_preds, ft_preds):
    if bp != gold or fp != gold:
        any_miss = True
        print(f"  {gold:9} | {bp:9} | {fp:9} | {text}")
if not any_miss:
    print("  (none — both models matched every held-out label)")

base     : 100.0%  (gpt-4o-mini-2024-07-18)
fine-tune: 100.0%  (ft:gpt-4o-mini-2024-07-18:personal::DuwBSoJZ)

misclassifications (gold | base | ft | text):
  (none — both models matched every held-out label)


## 12. Activate the fine-tune (optional)

To run the live app on your fine-tune, paste the `ft:...` id into `.env` as `BOOKING_ADVISOR_MODEL`,
then re-run the end-to-end eval to measure the effect in context:

```powershell
python tests/run_evals.py
```

`booking_advisor.py` falls back to the base model if the id is ever unreachable, so a stale id never
breaks booking.